In [ ]:
# Script to generate the SystemVerilog NOP insertion code
instructions = [
    ("32'h01000093", "// ADDI x1, x0, 16       x1 = 16"),
    ("32'h00800113", "// ADDI x2, x0, 8        x2 = 8"),
    ("32'hFFF00193", "// ADDI x3, x0, -1       x3 = 0xFFFFFFFF"),
    ("32'h80000237", "// LUI  x4, 0x80000      x4 = 0x80000000"),
    ("32'h002082B3", "// ADD  x5, x1, x2       x5 = 16 + 8 = 24 (0x18)"),
    ("32'h40208333", "// SUB  x6, x1, x2       x6 = 16 - 8 = 8"),
    ("32'h00508393", "// ADDI x7, x1, 5        x7 = 16 + 5 = 21 (0x15)"),
    ("32'hFFD08413", "// ADDI x8, x1, -3       x8 = 16 - 3 = 13 (0x0D)"),
    ("32'h0020F4B3", "// AND  x9, x1, x2       x9 = 0x10 & 0x08 = 0x00"),
    ("32'h0020E533", "// OR   x10, x1, x2      x10 = 0x10 | 0x08 = 0x18"),
    ("32'h0020C5B3", "// XOR  x11, x1, x2      x11 = 0x10 ^ 0x08 = 0x18"),
    ("32'h00F0F613", "// ANDI x12, x1, 15      x12 = 0x10 & 0x0F = 0x00"),
    ("32'h00F0E693", "// ORI  x13, x1, 15      x13 = 0x10 | 0x0F = 0x1F"),
    ("32'h0FF0C713", "// XORI x14, x1, 255     x14 = 0x10 ^ 0xFF = 0xEF"),
    ("32'h0020A7B3", "// SLT  x15, x1, x2      x15 = (16 < 8) ? 1 : 0 = 0"),
    ("32'h0030A833", "// SLT  x16, x1, x3      x16 = (16 < -1) ? 1 : 0 = 0"),
    ("32'h0020B8B3", "// SLTU x17, x1, x2      x17 = (16 < 8) ? 1 : 0 = 0"),
    ("32'h0030B933", "// SLTU x18, x1, x3      x18 = (16 < 0xFFFFFFFF) ? 1 : 0 = 1"),
    ("32'h0080A993", "// SLTI x19, x1, 8       x19 = (16 < 8) ? 1 : 0 = 0"),
    ("32'h0140AA13", "// SLTI x20, x1, 20      x20 = (16 < 20) ? 1 : 0 = 1"),
    ("32'h0080BA93", "// SLTIU x21, x1, 8      x21 = (16 < 8) ? 1 : 0 = 0"),
    ("32'h0140BB13", "// SLTIU x22, x1, 20     x22 = (16 < 20) ? 1 : 0 = 1"),
    ("32'h00209B93", "// SLLI x23, x1, 2       x23 = 16 << 2 = 64 (0x40)"),
    ("32'h00111C13", "// SLLI x24, x2, 1       x24 = 8 << 1 = 16 (0x10)"),
    ("32'h00115C93", "// SRLI x25, x2, 1       x25 = 8 >> 1 = 4"),
    ("32'h0021DD33", "// SRL  x26, x3, x2      x26 = 0xFFFFFFFF >> 8 = 0x00FFFFFF"),
    ("32'h4021DD93", "// SRAI x27, x3, 2       x27 = -1 >>> 2 = -1 (0xFFFFFFFF)"),
    ("32'h00221E13", "// SLL  x28, x4, x2      x28 = 0x80000000 << 8 = 0x00000000"),
    ("32'h12345EB7", "// LUI  x29, 0x12345     x29 = 0x12345000"),
    ("32'h00000F17", "// AUIPC x30, 0          x30 = PC + 0 (address of this instruction)"),
    ("32'h80008FB7", "// LUI  x31, 0x80008     x31 = 0x80008000"),
    ("32'hFFF08F93", "// ADDI x31, x1, -1      x31 = 16 - 1 = 15 (0x0F)"),
    ("32'h00100073", "// EBREAK")
]

output_lines = []
inst_idx = 0
mem_idx = 0

while inst_idx < len(instructions):
    # Check if current memory index is "Bad" (Ends in 11 binary -> 3 mod 4? No, 7 mod 8)
    # Byte Addr: ...11100 (0x1C).
    # Word Addr: ...111 (7).
    # Mod 8 == 7?
    
    if (mem_idx % 8) == 7:
        output_lines.append(f"        test_program[{mem_idx}] = 32'h00000013; // NOP (Skip bad address suffix ...1C)")
    else:
        code, comment = instructions[inst_idx]
        output_lines.append(f"        test_program[{mem_idx}] = {code}; {comment}")
        inst_idx += 1
    
    mem_idx += 1

# Print the SystemVerilog block
print("// Generated Test Program Load with NOP inserts")
print(f"        test_program = new[{mem_idx}];")
for line in output_lines:
    print(line)
